# ML Assignment 1: final polynomial models

**Tanmay Dixit | BT2024016**

This notebook reproduces the selected models and final submission files from
your four original CSVs. The degree and regularization choices have already
been selected with development-only cross-validation. It does not repeat the
entire model search. The project ZIP includes that search code and evidence.

The selected degree is **5 for var1** and **12 for var2**. Each prediction is a
fixed average of two polynomial fits, which is itself polynomial. Previously
measured holdout MSEs are 0.278873 and 0.274358. These are validation results;
the hidden-test scores are unknown.

Run the cells from top to bottom. A CPU runtime is sufficient.


## 1. Check the runtime and install the PDF dependency

This preserves the scientific packages already supplied by Colab. Only a
missing ReportLab package is installed. Do not run the old NumPy installation
cell. If an earlier installation has broken imports, save your files and use a
fresh runtime before running this notebook.


In [ ]:
import importlib.util
import subprocess
import sys

try:
    import numpy as np
    import pandas as pd
    import scipy
    import sklearn
    import matplotlib
    from sklearn.linear_model import ARDRegression, LassoLars, Ridge
    from threadpoolctl import threadpool_limits
except (ImportError, ValueError, AttributeError) as exc:
    raise RuntimeError(
        "Scientific imports failed. In Colab, save needed runtime files, use "
        "Runtime > Disconnect and delete runtime, reconnect, and run this "
        "updated notebook from the top. Upload the CSVs again."
    ) from exc

if importlib.util.find_spec("reportlab") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "reportlab==4.4.9"])
import reportlab
print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| scikit-learn", sklearn.__version__)
print("ReportLab", reportlab.Version)
print("Setup complete.")


## 2. Your identity and datasets

Upload all four original files. Keep their filenames. The final configuration
checks the training files against their recorded hashes so that old validation
scores cannot be attached to different data.


In [ ]:
from pathlib import Path
from datetime import datetime

ROLL = "BT2024016"
STUDENT_NAME = "Tanmay Dixit"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
required = [f"{ROLL}_{split}_var{p}.csv" for p in (1, 2) for split in ("train", "test")]
try:
    from google.colab import files
except ImportError:
    print("Local Jupyter: put the four CSVs in", DATA_DIR.resolve())
else:
    if any(not (DATA_DIR / name).is_file() for name in required):
        for name, content in files.upload().items():
            (DATA_DIR / Path(name).name).write_bytes(content)
missing = [name for name in required if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("Missing datasets: " + ", ".join(missing))
print("All four datasets are present.")


## 3. Polynomial terms, fitting and portable JSON inference. Scaling is fitted on training rows only.

This cell writes the Python module used by the following cells. The full source is visible below.

In [ ]:
%%writefile polynomial_model.py
"""Fit and serialize polynomial regressors without pickled estimator objects."""
from __future__ import annotations

import json
from pathlib import Path
import numpy as np
from sklearn.linear_model import ARDRegression, LassoLars, LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


def design_matrix(x, powers, basis="monomial"):
    """Products of univariate polynomials, with total degree bounded by powers."""
    x = np.asarray(x, dtype=float)
    powers = np.asarray(powers, dtype=int)
    if not len(powers):
        return np.empty((len(x), 0))
    degree = int(powers.max())
    out = np.ones((len(x), len(powers)))
    for j in range(x.shape[1]):
        if basis == "legendre":
            vander = np.polynomial.legendre.legvander(x[:, j], degree)
        elif basis == "monomial":
            vander = np.polynomial.polynomial.polyvander(x[:, j], degree)
        else:
            raise ValueError(f"Unknown polynomial basis: {basis}")
        out *= vander[:, powers[:, j]]
    return out


def predict_model(model, x):
    x = np.asarray(x, dtype=float)
    if x.ndim != 2 or x.shape[1] != len(model["features"]):
        raise ValueError("Input feature count does not match the fitted model.")
    if "components" in model:
        predictions = [predict_model(component, x) for component in model["components"]]
        return np.average(predictions, axis=0, weights=model["weights"])
    shifted = (x - np.asarray(model["input_mean"])) / np.asarray(model["input_scale"])
    z = design_matrix(shifted, model["powers"], model["basis"])
    result = z @ np.asarray(model["coefficients"]) + model["intercept"]
    if not np.isfinite(result).all():
        raise ValueError("Non-finite predictions.")
    return result


def fit_model(x, y, features, spec):
    """Fit preprocessing and regression using only the provided training rows."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if "components" in spec:
        models = [fit_model(x, y, features, part) for part in spec["components"]]
        return {"schema_version": 1, "features": list(features), "degree": max(m["degree"] for m in models),
                "spec": spec, "components": models, "weights": spec.get("weights", [1.] * len(models)),
                "fit_rows": len(x)}
    input_mean = np.zeros(x.shape[1]); input_scale = np.ones(x.shape[1])
    if spec.get("input_standardize", False):
        ss = StandardScaler().fit(x)
        input_mean, input_scale = ss.mean_, ss.scale_
    shifted = (x-input_mean) / input_scale
    powers = PolynomialFeatures(spec["degree"], include_bias=False).fit(shifted).powers_
    basis = spec.get("basis", "monomial")
    z = design_matrix(shifted, powers, basis)
    scaler = StandardScaler().fit(z)
    standardized = scaler.transform(z)
    weights = powers.sum(axis=1).astype(float) ** (-spec.get("penalty", 0.))
    method = spec.get("method", "ridge")
    alpha = spec.get("alpha", 0.)
    if method == "ols":
        estimator = LinearRegression()
    elif method == "ridge":
        estimator = Ridge(alpha=alpha, solver="svd")
    elif method == "lasso":
        estimator = LassoLars(alpha=alpha, max_iter=2000)
    elif method == "ard":
        estimator = ARDRegression(max_iter=500, tol=1e-4)
    else:
        raise ValueError(f"Unknown fitting method: {method}")
    estimator.fit(standardized*weights, y)
    coef = estimator.coef_ * weights
    intercept = float(estimator.intercept_)
    relax = spec.get("relax", 0.)
    if relax:
        selected = np.abs(coef) > 1e-9
        if np.any(selected):
            refit = Ridge(alpha=.01, solver="svd").fit(standardized[:, selected], y)
            adjusted = np.zeros_like(coef)
            adjusted[selected] = refit.coef_
            coef = (1-relax)*coef + relax*adjusted
            intercept = (1-relax)*intercept + relax*float(refit.intercept_)
    raw_coef = coef / scaler.scale_
    raw_intercept = intercept - float(raw_coef @ scaler.mean_)
    active = raw_coef != 0
    model = {"schema_version": 1, "features": list(features), "degree": int(spec["degree"]),
             "basis": basis, "spec": dict(spec), "input_mean": input_mean.tolist(),
             "input_scale": input_scale.tolist(), "powers": powers[active].tolist(),
             "coefficients": raw_coef[active].tolist(), "intercept": raw_intercept,
             "fit_rows": len(x), "candidate_terms": len(powers), "active_terms": int(active.sum())}
    expected = standardized@coef+intercept
    np.testing.assert_allclose(predict_model(model,x), expected, rtol=1e-8, atol=1e-8)
    return model


def save_model(model, path):
    Path(path).write_text(json.dumps(model,indent=2,allow_nan=False),encoding="utf-8")


def load_model(path):
    model=json.loads(Path(path).read_text(encoding="utf-8"))
    if model.get("schema_version") != 1:
        raise ValueError("Unsupported model file version.")
    return model


## 4. Create the three-page PDF from measured metrics and diagnostic plots.

This cell writes the Python module used by the following cells. The full source is visible below.

In [ ]:
%%writefile report.py
"""Create the assignment report from measured results."""
from pathlib import Path
from xml.sax.saxutils import escape
import json
import pandas as pd
import numpy as np

CV = pd.DataFrame(json.loads('[{"problem":1,"candidate":"lasso5_005","degree":5,"cv_mse":0.3069783882,"fold_mse_sd":0.0286804056,"mean_active_terms":162.8666666667,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5, \\"alpha\\": 0.005, \\"relax\\": 0.0}","fold_1_mse":0.2919284709,"fold_2_mse":0.2574978761,"fold_3_mse":0.3171859095,"fold_4_mse":0.3574285246,"fold_5_mse":0.3068051436,"fold_6_mse":0.305175708,"fold_7_mse":0.2838473418,"fold_8_mse":0.3155298125,"fold_9_mse":0.3406678251,"fold_10_mse":0.2981605674,"fold_11_mse":0.2809439382,"fold_12_mse":0.2618000561,"fold_13_mse":0.3236790769,"fold_14_mse":0.3396375832,"fold_15_mse":0.3243879885},{"problem":1,"candidate":"lasso5_008","degree":5,"cv_mse":0.3015181389,"fold_mse_sd":0.0264967651,"mean_active_terms":132.3333333333,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5, \\"alpha\\": 0.008, \\"relax\\": 0.0}","fold_1_mse":0.2835496456,"fold_2_mse":0.2639936303,"fold_3_mse":0.3206975151,"fold_4_mse":0.3535447455,"fold_5_mse":0.2938955917,"fold_6_mse":0.2998501599,"fold_7_mse":0.2763363458,"fold_8_mse":0.3067682025,"fold_9_mse":0.3306763196,"fold_10_mse":0.2922639366,"fold_11_mse":0.2784228001,"fold_12_mse":0.2604758077,"fold_13_mse":0.3250179365,"fold_14_mse":0.3207267842,"fold_15_mse":0.316552663},{"problem":1,"candidate":"lasso5_010","degree":5,"cv_mse":0.303916564,"fold_mse_sd":0.0262792254,"mean_active_terms":119.8666666667,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5, \\"alpha\\": 0.01, \\"relax\\": 0.0}","fold_1_mse":0.2805556254,"fold_2_mse":0.2723285389,"fold_3_mse":0.3254093119,"fold_4_mse":0.3570146662,"fold_5_mse":0.2904853889,"fold_6_mse":0.3012101434,"fold_7_mse":0.2779121708,"fold_8_mse":0.308821793,"fold_9_mse":0.32797692,"fold_10_mse":0.2969560179,"fold_11_mse":0.2829649696,"fold_12_mse":0.2639262245,"fold_13_mse":0.3286822578,"fold_14_mse":0.320803949,"fold_15_mse":0.3237004825},{"problem":1,"candidate":"relaxed5_015","degree":5,"cv_mse":0.3020244903,"fold_mse_sd":0.0240945511,"mean_active_terms":99.5333333333,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5, \\"alpha\\": 0.015, \\"relax\\": 0.5}","fold_1_mse":0.2852130178,"fold_2_mse":0.2716009686,"fold_3_mse":0.3167938874,"fold_4_mse":0.3515578121,"fold_5_mse":0.2967817515,"fold_6_mse":0.3014751199,"fold_7_mse":0.2683020832,"fold_8_mse":0.3101026035,"fold_9_mse":0.3232699583,"fold_10_mse":0.2941709129,"fold_11_mse":0.2826136751,"fold_12_mse":0.2685671447,"fold_13_mse":0.3206043881,"fold_14_mse":0.324199624,"fold_15_mse":0.3151144071},{"problem":1,"candidate":"relaxed5_020","degree":5,"cv_mse":0.3072887509,"fold_mse_sd":0.0251544955,"mean_active_terms":89.6666666667,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5, \\"alpha\\": 0.02, \\"relax\\": 0.5}","fold_1_mse":0.2853194418,"fold_2_mse":0.2749473838,"fold_3_mse":0.3339333162,"fold_4_mse":0.3598428891,"fold_5_mse":0.2914380972,"fold_6_mse":0.3072839082,"fold_7_mse":0.288873635,"fold_8_mse":0.3126863139,"fold_9_mse":0.3128692676,"fold_10_mse":0.2988636898,"fold_11_mse":0.285929385,"fold_12_mse":0.272294394,"fold_13_mse":0.3316983567,"fold_14_mse":0.3193263334,"fold_15_mse":0.3340248519},{"problem":1,"candidate":"refit5_030","degree":5,"cv_mse":0.320594226,"fold_mse_sd":0.035688642,"mean_active_terms":75.8666666667,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5, \\"alpha\\": 0.03, \\"relax\\": 1.0}","fold_1_mse":0.2934249938,"fold_2_mse":0.2772572476,"fold_3_mse":0.3210364444,"fold_4_mse":0.3718785031,"fold_5_mse":0.3112280157,"fold_6_mse":0.3243809709,"fold_7_mse":0.2601417912,"fold_8_mse":0.3193293014,"fold_9_mse":0.3561084771,"fold_10_mse":0.3120297404,"fold_11_mse":0.2803631558,"fold_12_mse":0.3003767313,"fold_13_mse":0.3403718908,"fold_14_mse":0.3773476286,"fold_15_mse":0.3636384979},{"problem":1,"candidate":"lasso6_010","degree":6,"cv_mse":0.3245725968,"fold_mse_sd":0.0292792487,"mean_active_terms":152.0,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 6, \\"alpha\\": 0.01, \\"relax\\": 0.0}","fold_1_mse":0.296768533,"fold_2_mse":0.309357634,"fold_3_mse":0.342505176,"fold_4_mse":0.3562136518,"fold_5_mse":0.303838618,"fold_6_mse":0.3227841927,"fold_7_mse":0.2987390434,"fold_8_mse":0.3254085213,"fold_9_mse":0.3646394071,"fold_10_mse":0.3123079968,"fold_11_mse":0.310764008,"fold_12_mse":0.2706754024,"fold_13_mse":0.3278334878,"fold_14_mse":0.3446699037,"fold_15_mse":0.3820833767},{"problem":1,"candidate":"ard5","degree":5,"cv_mse":0.3154109855,"fold_mse_sd":0.0299541447,"mean_active_terms":109.6,"components":1,"spec":"{\\"method\\": \\"ard\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5}","fold_1_mse":0.3150344126,"fold_2_mse":0.284177117,"fold_3_mse":0.3244278578,"fold_4_mse":0.3729315223,"fold_5_mse":0.339666811,"fold_6_mse":0.3220688396,"fold_7_mse":0.2873840405,"fold_8_mse":0.2952856402,"fold_9_mse":0.3241410702,"fold_10_mse":0.2918800772,"fold_11_mse":0.2768501962,"fold_12_mse":0.2834028517,"fold_13_mse":0.3394432904,"fold_14_mse":0.366400071,"fold_15_mse":0.3080709846},{"problem":1,"candidate":"ridge5_legendre","degree":5,"cv_mse":0.4575586938,"fold_mse_sd":0.0546523873,"mean_active_terms":461.0,"components":1,"spec":"{\\"method\\": \\"ridge\\", \\"basis\\": \\"legendre\\", \\"degree\\": 5, \\"alpha\\": 0.1, \\"penalty\\": 2.0}","fold_1_mse":0.4887354618,"fold_2_mse":0.399244041,"fold_3_mse":0.3984327318,"fold_4_mse":0.5285261207,"fold_5_mse":0.4687362931,"fold_6_mse":0.4398902347,"fold_7_mse":0.4180094949,"fold_8_mse":0.4370696439,"fold_9_mse":0.5335329455,"fold_10_mse":0.5243029983,"fold_11_mse":0.4429901207,"fold_12_mse":0.3676579165,"fold_13_mse":0.4744672211,"fold_14_mse":0.4103628306,"fold_15_mse":0.5314223521},{"problem":1,"candidate":"average_lasso_ard","degree":5,"cv_mse":0.2993067544,"fold_mse_sd":0.0270330634,"mean_active_terms":null,"components":2,"spec":"{\\"components\\": [{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5, \\"alpha\\": 0.008, \\"relax\\": 0.0}, {\\"method\\": \\"ard\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5}], \\"weights\\": [1.0, 1.0]}","fold_1_mse":0.2930911912,"fold_2_mse":0.2580649326,"fold_3_mse":0.315521459,"fold_4_mse":0.35299085,"fold_5_mse":0.3076912325,"fold_6_mse":0.3042943883,"fold_7_mse":0.2753453578,"fold_8_mse":0.2922179353,"fold_9_mse":0.3175714957,"fold_10_mse":0.2847587027,"fold_11_mse":0.2719394101,"fold_12_mse":0.2597487124,"fold_13_mse":0.3200533439,"fold_14_mse":0.3358612108,"fold_15_mse":0.3004510934},{"problem":1,"candidate":"average_lasso_relaxed","degree":5,"cv_mse":0.3007870549,"fold_mse_sd":0.0252080741,"mean_active_terms":null,"components":2,"spec":"{\\"components\\": [{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5, \\"alpha\\": 0.008, \\"relax\\": 0.0}, {\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 5, \\"alpha\\": 0.015, \\"relax\\": 0.5}], \\"weights\\": [1.0, 1.0]}","fold_1_mse":0.2834795946,"fold_2_mse":0.2665209025,"fold_3_mse":0.3183113632,"fold_4_mse":0.3515008302,"fold_5_mse":0.2943383529,"fold_6_mse":0.2999791173,"fold_7_mse":0.2708997958,"fold_8_mse":0.3076598227,"fold_9_mse":0.3252100571,"fold_10_mse":0.292119904,"fold_11_mse":0.2799455723,"fold_12_mse":0.2636049497,"fold_13_mse":0.3219107945,"fold_14_mse":0.3215041775,"fold_15_mse":0.3148205886},{"problem":2,"candidate":"ridge12_legendre","degree":12,"cv_mse":0.3087279403,"fold_mse_sd":0.0543460151,"mean_active_terms":454.0,"components":1,"spec":"{\\"method\\": \\"ridge\\", \\"basis\\": \\"legendre\\", \\"degree\\": 12, \\"alpha\\": 0.01, \\"penalty\\": 2.0}","fold_1_mse":0.2655580733,"fold_2_mse":0.33935423,"fold_3_mse":0.2445208154,"fold_4_mse":0.3903298688,"fold_5_mse":0.2781415721,"fold_6_mse":0.252134514,"fold_7_mse":0.3223589048,"fold_8_mse":0.3162046013,"fold_9_mse":0.3265986868,"fold_10_mse":0.264465669,"fold_11_mse":0.3159647496,"fold_12_mse":0.2463245672,"fold_13_mse":0.274070893,"fold_14_mse":0.3870101675,"fold_15_mse":0.4078817914},{"problem":2,"candidate":"ridge14_legendre","degree":14,"cv_mse":0.3105639407,"fold_mse_sd":0.0533516779,"mean_active_terms":679.0,"components":1,"spec":"{\\"method\\": \\"ridge\\", \\"basis\\": \\"legendre\\", \\"degree\\": 14, \\"alpha\\": 0.01, \\"penalty\\": 2.0}","fold_1_mse":0.2650433037,"fold_2_mse":0.3413377252,"fold_3_mse":0.2484423993,"fold_4_mse":0.3966312281,"fold_5_mse":0.2836406264,"fold_6_mse":0.2509700354,"fold_7_mse":0.3270917629,"fold_8_mse":0.3297975715,"fold_9_mse":0.3230962956,"fold_10_mse":0.2582145371,"fold_11_mse":0.3250215724,"fold_12_mse":0.2501778306,"fold_13_mse":0.2784473569,"fold_14_mse":0.3990111967,"fold_15_mse":0.3815356686},{"problem":2,"candidate":"ridge11_legendre","degree":11,"cv_mse":0.3068407697,"fold_mse_sd":0.0588617909,"mean_active_terms":363.0,"components":1,"spec":"{\\"method\\": \\"ridge\\", \\"basis\\": \\"legendre\\", \\"degree\\": 11, \\"alpha\\": 0.01, \\"penalty\\": 2.0}","fold_1_mse":0.2749794488,"fold_2_mse":0.3345561038,"fold_3_mse":0.2508521496,"fold_4_mse":0.4284439643,"fold_5_mse":0.2649007954,"fold_6_mse":0.2612186508,"fold_7_mse":0.297082583,"fold_8_mse":0.3063751652,"fold_9_mse":0.311441311,"fold_10_mse":0.267947703,"fold_11_mse":0.3099453119,"fold_12_mse":0.2375388311,"fold_13_mse":0.2620019935,"fold_14_mse":0.3953790584,"fold_15_mse":0.3999484758},{"problem":2,"candidate":"ridge8_legendre","degree":8,"cv_mse":0.3121183958,"fold_mse_sd":0.062212387,"mean_active_terms":164.0,"components":1,"spec":"{\\"method\\": \\"ridge\\", \\"basis\\": \\"legendre\\", \\"degree\\": 8, \\"alpha\\": 0.31622776601683794, \\"penalty\\": 1.0}","fold_1_mse":0.3513603982,"fold_2_mse":0.317113254,"fold_3_mse":0.2454694092,"fold_4_mse":0.3853351565,"fold_5_mse":0.287026264,"fold_6_mse":0.2552466784,"fold_7_mse":0.3092866,"fold_8_mse":0.328101977,"fold_9_mse":0.3126000063,"fold_10_mse":0.2579720849,"fold_11_mse":0.3055230198,"fold_12_mse":0.224705241,"fold_13_mse":0.2691682203,"fold_14_mse":0.3691822241,"fold_15_mse":0.463685403},{"problem":2,"candidate":"relaxed9_001","degree":9,"cv_mse":0.2910132878,"fold_mse_sd":0.0458319864,"mean_active_terms":118.4,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 9, \\"alpha\\": 0.001, \\"relax\\": 0.5}","fold_1_mse":0.3744329409,"fold_2_mse":0.2955711402,"fold_3_mse":0.2452280455,"fold_4_mse":0.3322672531,"fold_5_mse":0.2959900589,"fold_6_mse":0.2512928109,"fold_7_mse":0.2503124335,"fold_8_mse":0.2710319565,"fold_9_mse":0.365715683,"fold_10_mse":0.2701157558,"fold_11_mse":0.2952221516,"fold_12_mse":0.2128935475,"fold_13_mse":0.2696512237,"fold_14_mse":0.2959464018,"fold_15_mse":0.3395279135},{"problem":2,"candidate":"refit12_003","degree":12,"cv_mse":0.3172990198,"fold_mse_sd":0.0650980614,"mean_active_terms":113.6666666667,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 12, \\"alpha\\": 0.003, \\"relax\\": 1.0}","fold_1_mse":0.3153051395,"fold_2_mse":0.3002280145,"fold_3_mse":0.2447129308,"fold_4_mse":0.4541948664,"fold_5_mse":0.2772486359,"fold_6_mse":0.2487434837,"fold_7_mse":0.3552387598,"fold_8_mse":0.3117604907,"fold_9_mse":0.3801691263,"fold_10_mse":0.2582420054,"fold_11_mse":0.2951551959,"fold_12_mse":0.2465581067,"fold_13_mse":0.2788168174,"fold_14_mse":0.4081925218,"fold_15_mse":0.3849192018},{"problem":2,"candidate":"refit8_legendre","degree":8,"cv_mse":0.3224347181,"fold_mse_sd":0.0424281327,"mean_active_terms":125.4666666667,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"legendre\\", \\"degree\\": 8, \\"alpha\\": 0.02, \\"relax\\": 1.0}","fold_1_mse":0.3011808033,"fold_2_mse":0.3583822879,"fold_3_mse":0.2613117629,"fold_4_mse":0.3510087804,"fold_5_mse":0.3255576065,"fold_6_mse":0.2634636141,"fold_7_mse":0.3686150274,"fold_8_mse":0.3805521929,"fold_9_mse":0.3521480337,"fold_10_mse":0.2846794336,"fold_11_mse":0.3136535667,"fold_12_mse":0.2728395933,"fold_13_mse":0.3178358611,"fold_14_mse":0.2966856968,"fold_15_mse":0.3886065115},{"problem":2,"candidate":"refit9_legendre","degree":9,"cv_mse":0.3161524675,"fold_mse_sd":0.042960612,"mean_active_terms":151.4666666667,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"legendre\\", \\"degree\\": 9, \\"alpha\\": 0.015, \\"relax\\": 1.0}","fold_1_mse":0.3061141571,"fold_2_mse":0.344716019,"fold_3_mse":0.2892561345,"fold_4_mse":0.3235761882,"fold_5_mse":0.3320420934,"fold_6_mse":0.2574625931,"fold_7_mse":0.3889499084,"fold_8_mse":0.3295930184,"fold_9_mse":0.3615008095,"fold_10_mse":0.2781345735,"fold_11_mse":0.3127287475,"fold_12_mse":0.2640181617,"fold_13_mse":0.2760451461,"fold_14_mse":0.2839596921,"fold_15_mse":0.39418977},{"problem":2,"candidate":"lasso9_0005","degree":9,"cv_mse":0.2943632112,"fold_mse_sd":0.0536243775,"mean_active_terms":133.5333333333,"components":1,"spec":"{\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 9, \\"alpha\\": 0.0005, \\"relax\\": 0.0}","fold_1_mse":0.3989550782,"fold_2_mse":0.2855857653,"fold_3_mse":0.239098202,"fold_4_mse":0.3578034928,"fold_5_mse":0.2858673363,"fold_6_mse":0.2438034147,"fold_7_mse":0.2556089525,"fold_8_mse":0.2707859041,"fold_9_mse":0.3654716112,"fold_10_mse":0.2717460252,"fold_11_mse":0.2915494559,"fold_12_mse":0.2117031195,"fold_13_mse":0.2678672884,"fold_14_mse":0.3113990051,"fold_15_mse":0.3582035167},{"problem":2,"candidate":"average_ridge_lasso","degree":12,"cv_mse":0.2781193008,"fold_mse_sd":0.0405011055,"mean_active_terms":null,"components":2,"spec":"{\\"components\\": [{\\"method\\": \\"ridge\\", \\"basis\\": \\"legendre\\", \\"degree\\": 12, \\"alpha\\": 0.01, \\"penalty\\": 2.0}, {\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 9, \\"alpha\\": 0.001, \\"relax\\": 0.5}], \\"weights\\": [1.0, 1.0]}","fold_1_mse":0.2934189474,"fold_2_mse":0.2982195845,"fold_3_mse":0.231522369,"fold_4_mse":0.3473083542,"fold_5_mse":0.2617697237,"fold_6_mse":0.2392375873,"fold_7_mse":0.2612596838,"fold_8_mse":0.2663058892,"fold_9_mse":0.3171001875,"fold_10_mse":0.2563166619,"fold_11_mse":0.2911974374,"fold_12_mse":0.2115222621,"fold_13_mse":0.2412373644,"fold_14_mse":0.3156420394,"fold_15_mse":0.3397314202},{"problem":2,"candidate":"average_three","degree":12,"cv_mse":0.2783585195,"fold_mse_sd":0.0358181442,"mean_active_terms":null,"components":3,"spec":"{\\"components\\": [{\\"method\\": \\"ridge\\", \\"basis\\": \\"legendre\\", \\"degree\\": 12, \\"alpha\\": 0.01, \\"penalty\\": 2.0}, {\\"method\\": \\"lasso\\", \\"basis\\": \\"monomial\\", \\"degree\\": 9, \\"alpha\\": 0.001, \\"relax\\": 0.5}, {\\"method\\": \\"lasso\\", \\"basis\\": \\"legendre\\", \\"degree\\": 8, \\"alpha\\": 0.02, \\"relax\\": 1.0}], \\"weights\\": [1.0, 1.0, 1.0]}","fold_1_mse":0.2733903009,"fold_2_mse":0.3012883355,"fold_3_mse":0.2317873222,"fold_4_mse":0.3327950832,"fold_5_mse":0.2743575099,"fold_6_mse":0.2371577511,"fold_7_mse":0.2850019556,"fold_8_mse":0.283673257,"fold_9_mse":0.3045756188,"fold_10_mse":0.2549736474,"fold_11_mse":0.2911747773,"fold_12_mse":0.2153235993,"fold_13_mse":0.2521057998,"fold_14_mse":0.2949436597,"fold_15_mse":0.342829175}]'))
REPO = 'https://github.com/tdixit547/ml-assignment-1-bt2024016'


def make_diagnostics(p, out_dir):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    df = pd.read_csv(out_dir/f'var{p}_holdout_predictions.csv')
    y, pred = df.actual_y.to_numpy(), df.predicted_y.to_numpy()
    residual = y-pred
    fig, axes = plt.subplots(1,3,figsize=(10,2.1))
    for ax in axes:
        ax.spines[['top','right']].set_visible(False)
        ax.tick_params(labelsize=8)
    axes[0].scatter(y,pred,s=8,alpha=.6,color='#a3343e')
    lo,hi=min(y.min(),pred.min()),max(y.max(),pred.max())
    axes[0].plot([lo,hi],[lo,hi],'--',color='#666666',lw=1)
    axes[0].set(title='Holdout predictions',xlabel='Actual y',ylabel='Predicted y')
    axes[1].scatter(pred,residual,s=8,alpha=.6,color='#a3343e')
    axes[1].axhline(0,color='#666666',ls='--',lw=1)
    axes[1].set(title='Holdout residuals',xlabel='Predicted y',ylabel='Actual - predicted')
    axes[2].hist(residual,bins=20,color='#a3343e',edgecolor='white',lw=.4)
    axes[2].set(title='Residual distribution',xlabel='Residual',ylabel='Rows')
    fig.tight_layout(pad=.7)
    path=out_dir/f'var{p}_table_diagnostics.png'
    fig.savefig(path,dpi=210,bbox_inches='tight',facecolor='white')
    plt.close(fig)
    return path


def write_report(metadata, out_dir):
    import reportlab
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_LEFT
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.platypus import Image, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

    out_dir = Path(out_dir)
    roll = metadata['roll']
    path = out_dir / f'{roll}_report.pdf'
    font_dir = Path(reportlab.__file__).parent / 'fonts'
    for name, filename in [('ReportSans','Vera.ttf'),('ReportSans-Bold','VeraBd.ttf')]:
        if name not in pdfmetrics.getRegisteredFontNames():
            pdfmetrics.registerFont(TTFont(name,str(font_dir/filename)))
    pdfmetrics.registerFontFamily('ReportSans',normal='ReportSans',bold='ReportSans-Bold',
                                 italic='ReportSans',boldItalic='ReportSans-Bold')
    styles=getSampleStyleSheet()
    styles.add(ParagraphStyle('TitleMain',fontName='ReportSans-Bold',fontSize=22,leading=27,
                              textColor=colors.HexColor('#8B1E2D'),alignment=TA_LEFT,spaceAfter=12))
    styles.add(ParagraphStyle('Section',fontName='ReportSans-Bold',fontSize=11.5,leading=15,
                              textColor=colors.HexColor('#8B1E2D'),spaceBefore=11,spaceAfter=6))
    styles.add(ParagraphStyle('TextMain',fontName='ReportSans',fontSize=9.6,leading=13.3,spaceAfter=7))
    styles.add(ParagraphStyle('Cell',fontName='ReportSans',fontSize=7.4,leading=9.2))
    styles.add(ParagraphStyle('HeadCell',fontName='ReportSans-Bold',fontSize=7.4,leading=9.2,textColor=colors.white))
    styles.add(ParagraphStyle('Note',fontName='ReportSans',fontSize=7.8,leading=10,spaceAfter=5))
    story=[]
    def text(value,style='TextMain'):
        story.append(Paragraph(value,styles[style]))

    def comparison(p, chosen):
        sub=CV[CV.problem==p]
        rows=[['Model','Degree','Basis','Parameters','CV MSE','Fold SD']]
        selected=None
        for _, row in sub.iterrows():
            spec=json.loads(row.spec)
            if 'components' in spec:
                model='Equal average'
                bases=list(dict.fromkeys(c['basis'] for c in spec['components']))
                basis=' / '.join(bases)
                parts=[]
                for c in spec['components']:
                    name={'ridge':'Ridge','ard':'ARD','lasso':'Lasso'}[c['method']]
                    if c.get('relax')==.5: name='relaxed Lasso'
                    if c.get('relax')==1: name='selected-term refit'
                    label=f"{name} d{c['degree']}"
                    if 'alpha' in c: label+=f" alpha={c['alpha']:g}"
                    parts.append(label)
                params=' + '.join(parts)+'; equal weights, component settings above'
            else:
                basis=spec['basis']
                model={'ridge':'Ridge','ard':'ARD','lasso':'Lasso'}[spec['method']]
                if spec.get('relax')==.5: model='Relaxed Lasso'
                if spec.get('relax')==1: model='Selected-term refit'
                if spec['method']=='ard': params='max_iter=500; tol=0.0001'
                else:
                    params=f"alpha={spec['alpha']:g}"
                    if spec.get('penalty'): params+=f"; degree penalty={spec['penalty']:g}"
                    if spec.get('relax'): params+=f"; relax={spec['relax']:g}; refit alpha=0.01"
            if row.candidate==chosen:
                model+=' (selected)'
                selected=len(rows)
            rows.append([model,str(int(row.degree)),basis,params,f'{row.cv_mse:.6f}',f'{row.fold_mse_sd:.6f}'])
            folds=row[[f'fold_{i}_mse' for i in range(1,16)]].astype(float).to_numpy()
            assert np.isclose(folds.mean(),row.cv_mse)
        data=[[Paragraph(escape(c),styles['HeadCell' if i==0 else 'Cell']) for c in r] for i,r in enumerate(rows)]
        width=A4[0]-84
        t=Table(data,colWidths=[78,42,55,width-78-42-55-65-65,65,65],repeatRows=1)
        commands=[('BACKGROUND',(0,0),(-1,0),colors.HexColor('#8B1E2D')),
                  ('GRID',(0,0),(-1,-1),.35,colors.HexColor('#cccccc')),
                  ('VALIGN',(0,0),(-1,-1),'MIDDLE'),
                  ('LEFTPADDING',(0,0),(-1,-1),4),('RIGHTPADDING',(0,0),(-1,-1),4),
                  ('TOPPADDING',(0,0),(-1,-1),4),('BOTTOMPADDING',(0,0),(-1,-1),4)]
        commands.append(('BACKGROUND',(0,selected),(-1,selected),colors.HexColor('#fbe9eb')))
        t.setStyle(TableStyle(commands))
        story.append(t)
        story.append(Spacer(1,5))
        text('All 11 finalists use the same 15 development folds. Fold SD is descriptive dispersion. '
             'Degree is maximum total degree. The degree-penalty value p scales terms of degree k by k^-p. '
             'Relax is the weight on the selected-term Ridge refit; all expansions use fold-fitted standardisation.','Note')

    text('Polynomial Regression','TitleMain')
    text(f"ML Assignment 1 | {escape(metadata['student_name'])} | {escape(roll)}")
    text('Data and objective','Section')
    text('Separate polynomial regression models predict the Net Power Score (var1) and Thermal Anomaly '
         'Score (var2). Each supplied training file has 1,000 labelled rows. Each test file has 1,000 '
         'rows without target labels. All six inputs are used for var1 and all three for var2. '
         'The four files contain no missing values. Original test-row order is preserved.')
    text('Polynomial models','Section')
    text('Each term has a total degree equal to the sum of its exponents. For example, x1^2 x2 '
         'has total degree 3. The search includes powers and interactions, with degree capped at 10 '
         'for var1 and 20 for var2. Monomial and Legendre bases are compared. Both represent polynomials. '
         'A fixed average of polynomial fits is also a polynomial, with degree no greater than the '
         'largest component degree. No tree, kernel with non-polynomial terms, or neural network is used.')
    text('Model selection','Section')
    text('A seed-42 split gives 800 development rows and 200 holdout rows per '
         'problem. Broad searches on the development rows compare ordinary least squares, Ridge, '
         'Lasso, relaxed Lasso, and automatic relevance determination (ARD) regression. ARD learns '
         'a separate shrinkage level for each coefficient. Expanded columns are standardized using '
         'only the fitting fold. Sparse term selection and any coefficient refit also stay inside that fold.')
    text('The all-feature expanded Ridge search covers degrees 1-6 for var1 and 1-12 for var2. '
         'Further sparse searches test var1 degrees 4, 5, 6, 8, 10 and '
         'var2 degrees 6, 8, 10, 12, 16, 20, followed by a finer penalty grid near the leading degrees. '
         'Ridge basis and degree-penalty searches extend to degree 8 and 20, respectively. The complete '
         'grids and attempted numerical fits are recorded with the code.')
    text('Nine individual finalists and two fixed equal-weight averages are compared per problem. '
         'Each is evaluated with three five-fold partitions of the same 800 development rows '
         '(seeds 42, 137, 2026). The lowest mean MSE over these 15 folds determines the final choice. '
         'These are model-selection scores, not an unbiased final test estimate. Repeated folds overlap.')
    text('Validation and final fitting','Section')
    text('The chosen settings are fixed before the final holdout evaluation. The 200-row holdout '
         'was examined during preliminary experiments and excluded from subsequent tuning, so it '
         'is a reused validation set. After recording MSE and R-squared, the fixed models are '
         'refitted on all 1,000 labelled rows to create the submission predictions.')
    text('MSE is the mean squared prediction error; lower is better. R-squared is 1 - SSE/SST; '
         'closer to 1 is better. Hidden-test scores cannot be computed because test targets were not supplied.')
    text('GitHub repository','Section')
    text(f'<link href="{REPO}" color="#8B1E2D">{REPO}</link>')

    explanations={
        1:('Selected degree: 5. The model averages two degree-5 monomial fits with equal weights: '
           'Lasso (alpha = 0.008) and ARD (maximum 500 iterations, tolerance 0.0001). '
           'Both use all six features and start from 461 polynomial terms, excluding the intercept. '
           'Coefficient shrinkage limits the influence of unsupported powers and interactions.'),
        2:('Selected degree: 12. The model equally averages a degree-12 Legendre Ridge fit '
           '(alpha = 0.01) and a degree-9 monomial relaxed-Lasso fit (alpha = 0.001). '
           'For Ridge, a standardized term of total degree k is divided by k^2 before fitting, '
           'so high-degree standardized coefficients receive a stronger penalty. The relaxed-Lasso '
           'fit averages its Lasso coefficients with a Ridge refit (alpha = 0.01) on selected terms. '
           'The two expansions contain 454 and 219 terms, excluding intercepts.'),
    }
    for result in metadata['results']:
        p=result['problem']
        story.append(PageBreak())
        title='Steam turbine optimization' if p==1 else 'Thermal reservoir mapping'
        text(f'var{p}: {title}','TitleMain')
        text('Selected model','Section')
        text(explanations[p])
        text('Measured results','Section')
        text(f"Repeated development CV MSE: {result['repeated_cv_mse']:.6f}. "
             f"Individual fold MSE standard deviation: {result['cv_fold_sd']:.6f}. "
             'This dispersion is descriptive; it is not a confidence interval.')
        measured=result['holdout']
        text(f"Selected-model holdout MSE: {measured['mse']:.6f}. "
             f"Holdout R-squared: {measured['r2']:.6f}. "
             'The following table compares all 11 finalists on the same 15 development folds.')
        comparison(p,result['selected_name'])
        story.append(Image(str(make_diagnostics(p,out_dir)),width=A4[0]-84,height=108))
        text('Selection rationale and final predictions','Section')
        if p==1:
            text('Degree-5 Lasso had lower repeated CV MSE than the tested degree-6 Lasso and dense '
                 'degree-5 Ridge finalists. Averaging the degree-5 Lasso and ARD fits gave the lowest '
                 'mean CV MSE among the finalists. Its advantage over Lasso alone is small and '
                 'does not establish a statistically significant difference. Test inputs are more often '
                 'near the feature boundaries than training inputs, so random-split performance may '
                 'not fully represent hidden-test performance.')
        else:
            text('Degree-9 relaxed Lasso and degree-12 Ridge were competitive individual candidates. '
                 'Their fixed average had lower mean repeated CV MSE than either component and '
                 'was the best of the tested finalists. The resulting polynomial has maximum '
                 'total degree 12. Hidden-test performance remains unknown.')
        text(f"Final training uses all {result['final_fit_rows']:,} labelled rows. "
             f"{escape(result['prediction_file'])} contains {result['test_rows']:,} finite predictions "
             'under the single header y. The saved JSON model supports inference without retraining.')

    def footer(canvas,document):
        canvas.saveState(); canvas.setFont('ReportSans',8); canvas.setFillColor(colors.HexColor('#666666'))
        canvas.drawString(42,24,f'ML Assignment 1 | {roll}')
        canvas.drawRightString(A4[0]-42,24,str(document.page)); canvas.restoreState()
    doc=SimpleDocTemplate(str(path),pagesize=A4,leftMargin=42,rightMargin=42,topMargin=35,bottomMargin=38,
                          title=f'Polynomial Regression - {roll}',author=metadata['student_name'])
    doc.build(story,onFirstPage=footer,onLaterPages=footer)
    return path


## 5. Validate inputs, evaluate the fixed models, refit on all labelled rows and write the outputs.

This cell writes the Python module used by the following cells. The full source is visible below.

In [ ]:
%%writefile assignment.py
"""Train, evaluate and use the selected polynomial regression models."""
from __future__ import annotations
import argparse,csv,hashlib,importlib.metadata,json,re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_limits
from polynomial_model import fit_model,predict_model,save_model,load_model
from report import write_report

SEED=42
HOLDOUT_FRACTION=.2
FEATURES={1:[f"x{i}" for i in range(1,7)],2:["x1","x2","x3"]}
PROJECT_DIR=Path(__file__).resolve().parent

def read_dataset(path, expected_features, training=False):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing dataset: {path}")
    # Check before pandas can silently rename duplicate column headers.
    with path.open(newline="", encoding="utf-8-sig") as stream:
        header = next(csv.reader(stream), [])
    if len(set(header)) != len(header):
        raise ValueError(f"{path.name}: duplicate column names are not supported.")
    frame = pd.read_csv(path)
    required = list(expected_features) + (["y"] if training else [])
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing}; found {list(frame.columns)}")
    if frame.empty:
        raise ValueError(f"{path.name}: the dataset is empty.")
    for column in required:
        if not pd.api.types.is_numeric_dtype(frame[column]):
            raise ValueError(f"{path.name}: {column} must be numeric.")
        if not np.isfinite(frame[column].to_numpy(dtype=float)).all():
            raise ValueError(f"{path.name}: {column} contains missing or infinite values.")
    # Test y, if present as a placeholder, and unrelated ID columns are ignored.
    # Test rows are never sorted or dropped.
    return frame


def detect_roll(data_dir, roll=None):
    data_dir = Path(data_dir)
    if roll is None:
        rolls = sorted({p.name[:-len("_train_var1.csv")]
                        for p in data_dir.glob("*_train_var1.csv")})
        if len(rolls) != 1:
            raise ValueError("Place one student's four CSVs in data/ or provide --roll YOUR_ROLL.")
        roll = rolls[0]
    if not re.fullmatch(r"[A-Za-z0-9-]+", roll):
        raise ValueError("The roll number must contain only letters, digits, or hyphens.")
    paths = {problem: {
        split: data_dir / f"{roll}_{split}_var{problem}.csv"
        for split in ("train", "test")
    } for problem in (1, 2)}
    missing = [str(p) for files in paths.values() for p in files.values() if not p.is_file()]
    if missing:
        raise FileNotFoundError("Upload the four personalised datasets. Missing:\n" + "\n".join(missing))
    return roll, paths


def scores(y_true, prediction):
    if not np.isfinite(prediction).all():
        raise ValueError("Model predictions contain non-finite values.")
    # R2 has no meaningful denominator for a constant target.
    r2 = None if np.ptp(np.asarray(y_true, dtype=float)) == 0 else float(r2_score(y_true, prediction))
    return {"mse": float(mean_squared_error(y_true, prediction)), "r2": r2}



def save_predictions(prediction,path,expected_rows):
    prediction=np.asarray(prediction,dtype=float)
    if prediction.shape!=(expected_rows,) or not np.isfinite(prediction).all():
        raise ValueError("Expected one finite prediction per test row.")
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    pd.DataFrame({"y":prediction}).to_csv(path,index=False)
    check=pd.read_csv(path)
    if check.columns.tolist()!=["y"] or len(check)!=expected_rows:
        raise RuntimeError("Incorrect prediction CSV format.")
    return path


def diagnostic_plot(result,y,prediction,out_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    p=result["problem"]
    plt.rcParams.update({"font.size":10,"axes.spines.top":False,"axes.spines.right":False})
    fig,axes=plt.subplots(2,2,figsize=(9.35,6.),layout="constrained")
    comparisons=result.get("cv_comparison", [{"label":"Selected", "mse":result["repeated_cv_mse"]}])
    labels=[c["label"] for c in comparisons]
    values=[c["mse"] for c in comparisons]
    axes[0,0].barh(labels,values,color=["#276577" if "Average" in label else "#aaaeb3" for label in labels])
    axes[0,0].invert_yaxis()
    axes[0,0].set(title="Candidate comparison: 15 CV folds",xlabel="Mean development CV MSE")
    axes[0,0].tick_params(axis="y",labelsize=8)
    for i,v in enumerate(values):
        axes[0,0].text(v,i,f" {v:.4f}",ha="left",va="center",fontsize=8)
    axes[0,0].set_xlim(0,max(values)*1.3)
    axes[0,1].scatter(y,prediction,s=14,alpha=.7,color="#276577")
    lo=min(y.min(),prediction.min()); hi=max(y.max(),prediction.max())
    axes[0,1].plot([lo,hi],[lo,hi],"--",lw=1,color="#963144")
    axes[0,1].set(title="Selected holdout predictions",xlabel="Actual y",ylabel="Predicted y")
    residual=y-prediction
    axes[1,0].scatter(prediction,residual,s=14,alpha=.7,color="#276577")
    axes[1,0].axhline(0,color="#963144",lw=1,ls="--")
    axes[1,0].set(title="Selected holdout residuals",xlabel="Predicted y",ylabel="Actual - predicted")
    axes[1,1].hist(residual,bins=18,color="#276577",edgecolor="white")
    axes[1,1].set(title="Residual distribution",xlabel="Actual - predicted",ylabel="Rows")
    fig.savefig(Path(out_dir)/f"var{p}_diagnostics.png",dpi=180,facecolor="white")
    plt.close(fig)


def run_assignment(data_dir="data",roll="BT2024016",out_dir="outputs",student_name="Tanmay Dixit",
                   config_path=None):
    # Fail before training if the PDF dependency is unavailable.
    import reportlab
    config_path=Path(config_path) if config_path else PROJECT_DIR/"selected_config.json"
    config=json.loads(config_path.read_text(encoding="utf-8"))
    roll,paths=detect_roll(data_dir,roll)
    datasets={p:{s:read_dataset(path,FEATURES[p],training=(s=="train"))
                 for s,path in files.items()} for p,files in paths.items()}
    hashes={path.name:hashlib.sha256(path.read_bytes()).hexdigest() for files in paths.values() for path in files.values()}
    for p in (1,2):
        filename=paths[p]["train"].name
        if hashes[filename]!=config["input_sha256"].get(filename):
            raise ValueError("The training file differs from the data used for model selection. "
                             "Rerun the search for those data before using its validation scores.")
    out_dir=Path(out_dir)
    if out_dir.exists() and any(out_dir.iterdir()):
        raise FileExistsError(f"{out_dir} is not empty. Choose a new output directory.")
    out_dir.mkdir(parents=True,exist_ok=True)
    results=[]
    with threadpool_limits(limits=1):
        for p,data in datasets.items():
            train,test=data["train"],data["test"]
            dev,holdout=train_test_split(train,test_size=HOLDOUT_FRACTION,random_state=SEED)
            f=FEATURES[p]; selected=config["selection"][str(p)]
            print(f"Fitting var{p}: {selected['name']}...",flush=True)
            evaluator=fit_model(dev[f],dev.y,f,selected["spec"])
            hp=predict_model(evaluator,holdout[f])
            previous_spec={"method":"ols","basis":"monomial","degree":4 if p==1 else 5,
                           "input_standardize":True}
            previous=fit_model(dev[f],dev.y,f,previous_spec)
            bp=predict_model(previous,holdout[f])
            current_scores=scores(holdout.y,hp); previous_scores=scores(holdout.y,bp)
            # The settings remain fixed. All labelled rows may now be used.
            final_model=fit_model(train[f],train.y,f,selected["spec"])
            predictions=predict_model(final_model,test[f])
            pred_path=save_predictions(predictions,out_dir/f"{roll}_pred_var{p}.csv",len(test))
            model_path=out_dir/f"var{p}_model.json"
            save_model(final_model,model_path)
            np.testing.assert_allclose(predict_model(load_model(model_path),test[f]),predictions,rtol=1e-12,atol=1e-12)
            pd.DataFrame({"row_number_1_based":holdout.index.to_numpy()+1,"actual_y":holdout.y.to_numpy(),
                          "previous_y":bp,"predicted_y":hp,"residual":holdout.y.to_numpy()-hp}).to_csv(
                          out_dir/f"var{p}_holdout_predictions.csv",index=False)
            r={"problem":p,"features":f,"degree":final_model["degree"],"selected_name":selected["name"],
               "spec":selected["spec"],"repeated_cv_mse":selected["repeated_cv_mse"],
               "cv_fold_sd":selected["fold_mse_sd"],"holdout":current_scores,"previous_holdout":previous_scores,
               "train_rows":len(train),"development_rows":len(dev),"holdout_rows":len(holdout),
               "test_rows":len(test),"final_fit_rows":len(train),"prediction_file":pred_path.name,
               "model_file":model_path.name,
               "active_terms_per_component":[m["active_terms"] for m in final_model.get("components",[final_model])]}
            results.append(r)
            r["cv_comparison"]=config.get("comparison_summary",{}).get(str(p),[])
            if not r["cv_comparison"]:
                r.pop("cv_comparison")
            diagnostic_plot(r,holdout.y.to_numpy(),hp,out_dir)
            print(f"var{p}: holdout MSE {current_scores['mse']:.9f}, R2 {current_scores['r2']:.9f}",flush=True)
    metadata={"roll":roll,"student_name":student_name,"seed":SEED,"holdout_fraction":HOLDOUT_FRACTION,
              "selection_rule":config["selection_rule"],"cv_seeds":[42,137,2026],"cv_folds":5,
              "holdout_note":"Original 200-row holdout reused for comparison; excluded from all new tuning.",
              "input_sha256":hashes,"results":results,
              "versions":{p:importlib.metadata.version(p) for p in
                          ["numpy","pandas","scipy","scikit-learn","matplotlib","reportlab","threadpoolctl"]}}
    (out_dir/"metrics.json").write_text(json.dumps(metadata,indent=2,allow_nan=False),encoding="utf-8")
    # Metrics and predictions survive any report-rendering failure.
    report=write_report(metadata,out_dir)
    print(f"Created both predictions, portable models, diagnostics and {report.name}.",flush=True)
    return metadata


def predict_saved(model_path,test_path,output_path):
    model=load_model(model_path)
    frame=read_dataset(test_path,model["features"])
    return save_predictions(predict_model(model,frame[model["features"]]),output_path,len(frame))


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    commands=parser.add_subparsers(dest="action",required=True)
    train=commands.add_parser("train",help="Reproduce the selected model fits and final outputs")
    train.add_argument("--data-dir",default="data")
    train.add_argument("--roll",default="BT2024016")
    train.add_argument("--name",default="Tanmay Dixit")
    train.add_argument("--out",default="outputs")
    train.add_argument("--config",default=None)
    infer=commands.add_parser("predict",help="Infer with a saved JSON polynomial")
    infer.add_argument("--model",required=True); infer.add_argument("--test",required=True)
    infer.add_argument("--output",required=True)
    report=commands.add_parser("report",help="Regenerate only the report from saved metrics and plots")
    report.add_argument("--out",default="outputs")
    args=parser.parse_args()
    if args.action=="train":
        run_assignment(args.data_dir,args.roll,args.out,args.name,args.config)
    elif args.action=="predict":
        path=predict_saved(args.model,args.test,args.output); print(f"Saved {path}")
    else:
        out=Path(args.out); metadata=json.loads((out/"metrics.json").read_text())
        print(write_report(metadata,out))


if __name__=="__main__":
    main()


## 6. Frozen model configuration

var1 averages Lasso and ARD fits of degree 5. var2 averages degree-12 Ridge in a
Legendre basis and degree-9 relaxed Lasso in a monomial basis. A term's total
degree is the sum of its exponents. The average has maximum degree 12.

The recorded selection scores come from three five-fold partitions of 800
development rows. The same 200-row holdout as the earlier notebook is used for
the final comparison and is excluded from new tuning. The full search and
fixed finalist shortlist are available in the project ZIP.


In [ ]:
import json

CONFIG = {'selection_rule': 'Minimum mean MSE across 3 x 5 development folds among 11 frozen finalists per '
                   'problem.',
 'input_sha256': {'BT2024016_test_var1.csv': '25b5a13f794ca91aba9b7e5f0a64f6cb8d0ff8f7e37bb7e8406b2211aabbcd74',
                  'BT2024016_test_var2.csv': '33b21e105fecdf763a17a470015f34616c7202ff2c2393cb7023b7dde261d1a8',
                  'BT2024016_train_var1.csv': 'c3e1c2cc3f2994e1bd58b386db6317b95e5490fc8a864263de0647e69ec38dfe',
                  'BT2024016_train_var2.csv': '3385eaab7c15c0120d7d24b1d356e525bd92851471e9e456ee0b5383066725ef'},
 'selection': {'1': {'name': 'average_lasso_ard',
                     'spec': {'components': [{'method': 'lasso',
                                              'basis': 'monomial',
                                              'degree': 5,
                                              'alpha': 0.008,
                                              'relax': 0.0},
                                             {'method': 'ard', 'basis': 'monomial', 'degree': 5}],
                              'weights': [1.0, 1.0]},
                     'repeated_cv_mse': 0.2993067543815013,
                     'fold_mse_sd': 0.027033063449651527,
                     'cv_seeds': [42, 137, 2026],
                     'cv_folds': 5,
                     'development_rows': 800},
               '2': {'name': 'average_ridge_lasso',
                     'spec': {'components': [{'method': 'ridge',
                                              'basis': 'legendre',
                                              'degree': 12,
                                              'alpha': 0.01,
                                              'penalty': 2.0},
                                             {'method': 'lasso',
                                              'basis': 'monomial',
                                              'degree': 9,
                                              'alpha': 0.001,
                                              'relax': 0.5}],
                              'weights': [1.0, 1.0]},
                     'repeated_cv_mse': 0.2781193008012439,
                     'fold_mse_sd': 0.040501105501661624,
                     'cv_seeds': [42, 137, 2026],
                     'cv_folds': 5,
                     'development_rows': 800}},
 'comparison_summary': {'1': [{'label': 'Ridge d5', 'mse': 0.4575586937685373},
                              {'label': 'Lasso d6', 'mse': 0.3245725968341478},
                              {'label': 'Lasso d5', 'mse': 0.3015181389393213},
                              {'label': 'ARD d5', 'mse': 0.3154109854776059},
                              {'label': 'Average d5', 'mse': 0.2993067543815013}],
                        '2': [{'label': 'Ridge d8', 'mse': 0.3121183957804485},
                              {'label': 'Ridge d12', 'mse': 0.3087279402897833},
                              {'label': 'Relaxed Lasso d9', 'mse': 0.2910132877707156},
                              {'label': 'Average d12', 'mse': 0.2781193008012439}]}}
Path("selected_config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print("Selected settings loaded.")


## 7. Fit and create the final files

This evaluates the fixed models on the 200-row comparison set, then fits them
on all 1,000 training rows. It creates one prediction per test row. Each run
uses a new output directory to keep files from different runs separate.


In [ ]:
import importlib
import polynomial_model, report, assignment
importlib.reload(polynomial_model)
importlib.reload(report)
importlib.reload(assignment)

OUTPUT_DIR = Path("outputs_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
metadata = assignment.run_assignment(DATA_DIR, ROLL, OUTPUT_DIR, STUDENT_NAME)


## 8. Review the results

These are measured holdout scores, not hidden-test scores. The earlier holdout
was already evaluated in the previous workflow; the report explains this limit.


In [ ]:
from IPython.display import display, Image

rows = [{"problem": f"var{r['problem']}", "degree": r["degree"],
         "repeated_cv_mse": r["repeated_cv_mse"],
         "previous_holdout_mse": r["previous_holdout"]["mse"],
         "new_holdout_mse": r["holdout"]["mse"],
         "new_holdout_r2": r["holdout"]["r2"]} for r in metadata["results"]]
display(pd.DataFrame(rows))
for p in (1, 2):
    prediction = pd.read_csv(OUTPUT_DIR / f"{ROLL}_pred_var{p}.csv")
    assert prediction.columns.tolist() == ["y"]
    assert len(prediction) == len(pd.read_csv(DATA_DIR / f"{ROLL}_test_var{p}.csv"))
    assert np.isfinite(prediction.y).all()
    display(Image(filename=str(OUTPUT_DIR / f"var{p}_diagnostics.png"), width=850))
print("Prediction format checks passed.")
print("Report:", OUTPUT_DIR / f"{ROLL}_report.pdf")


## 9. Download the results

Extract the ZIP and submit the report and two prediction CSVs. Upload the
project's extracted source files to GitHub and provide the repository link.
Review your name, roll and report before submission.


In [ ]:
import shutil

archive = shutil.make_archive(str(OUTPUT_DIR) + "_results", "zip", root_dir=OUTPUT_DIR)
print("Results archive:", archive)
try:
    from google.colab import files
except ImportError:
    print("Local Jupyter: the archive is in the notebook folder.")
else:
    files.download(archive)


## 10. Inference without retraining

This optional cell reconstructs predictions from the saved JSON coefficients.
The saved feature order is used, and test-row order is preserved.


In [ ]:
for p in (1, 2):
    recreated = OUTPUT_DIR / f"recreated_var{p}.csv"
    assignment.predict_saved(OUTPUT_DIR / f"var{p}_model.json",
                             DATA_DIR / f"{ROLL}_test_var{p}.csv", recreated)
    np.testing.assert_allclose(pd.read_csv(recreated).y,
                               pd.read_csv(OUTPUT_DIR / f"{ROLL}_pred_var{p}.csv").y,
                               rtol=1e-12, atol=1e-12)
print("Saved-model inference reproduces both prediction files.")
